# 01 · Setup, data, and how we're scored

**Goal of this notebook:** get the data onto your machine, look at it with your own eyes, and understand *exactly* how the organizers score a submission, before we write a single line of steering code.

**Runs on:** your Mac, CPU only. No GPU needed yet.

**Where this file lives:** `notebooks/01_setup_and_data.ipynb` inside your fork of the organizers' repo. The first cell moves the working directory up to the repo root, so `clas/`, `evaluate.py` and `data/` all resolve.

## Step 1 · Point the notebook at the repo root

The organizers' code is a small package (`clas/`) plus scripts (`download_data.py`, `evaluate.py`). We'll **reuse their code instead of rewriting it**. That way our numbers are computed the same way the leaderboard computes them.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)
print("has clas/ package:", (ROOT / "clas").exists())

## Step 2 · Check your Hugging Face login

Three things we need are **gated** on Hugging Face: you have to click "Agree" on each page once, while logged in:

- `openlanguagedata/flores_plus` (training sentences)
- `meta-llama/Llama-3.1-8B-Instruct` (model #1; approval can take a few hours)
- `CohereLabs/tiny-aya-global` (model #2)

Then run `hf auth login` **in the terminal** (with a *read* token from huggingface.co/settings/tokens). This cell just confirms the login worked.

In [ ]:
from huggingface_hub import whoami
print("logged in as:", whoami()["name"])

## Step 3 · Download the data

`download_data.py` builds two folders:

| folder | what | size | what we use it for |
|---|---|---|---|
| `data/train/` | FLORES-200 dev+devtest: the **same ~2000 sentences translated into every language** | ~2009 lines / language | finding *where* language lives inside the model (next notebook) |
| `data/validation/` | CLaS-Bench: **70 open-ended questions, parallel across languages** | 70 lines / language | measuring how well our steering works |

The real test set (100 unseen questions/language) stays hidden; the organizers run our code on it in January.

This downloads 23 languages, so give it a few minutes. It skips files that already exist, so re-running is safe.

In [ ]:
!{sys.executable} download_data.py --out data

## Step 4 · Look at the validation questions

Key idea: line *i* in `en.txt` is the **same question** as line *i* in `de.txt`, `ja.txt`, and so on. So for a pair like `ar → de`, the model gets the Arabic question and should answer in German.

In [ ]:
import pandas as pd
from clas import languages as L
from clas.io_utils import load_questions, read_lines

LANGS = L.HIGH + L.LOW
questions = load_questions(Path("data/validation"), LANGS)
print({c: len(q) for c, q in questions.items()})

pd.set_option("display.max_colwidth", 120)
pd.DataFrame({c: questions[c][:3] for c in ["en", "de", "ar", "ja", "hi", "sw"]}).T

**What to notice:** these are open-ended "explain / describe" questions, so answers will be long (up to 1024 tokens). That matters later: steering has to *hold* the target language over a long generation, not just the first few words.

## Step 5 · Look at the FLORES training sentences

Same parallel idea, but these are short news/wiki sentences. Next notebook we'll push all ~2000 sentences of each language through the model, record its hidden states, and average them. The difference between, say, the "German average" and the "all-languages average" is our **steering vector** for German.

Because the sentences say the *same thing* in every language, the content cancels out in that difference and what's left is mostly **language identity**. That's why parallel data is so useful here.

In [ ]:
train = {c: read_lines(Path(f"data/train/{c}.txt")) for c in LANGS}
print({c: len(s) for c, s in train.items()})

for c in ["en", "de", "ru", "zh", "th", "mt"]:
    print(f"{c}: {train[c][0]}")

## Step 6 · Tracks → language pairs → workload

Let's count how many generations each track needs on the validation set. This tells us how much GPU time to budget.

In [ ]:
for track in ["t1", "t2"]:
    p = L.pairs(track)
    print(f"{track.upper()}: {len(p)} pairs × 70 questions = {len(p) * 70:,} generations per model")
print("first few T1 pairs:", L.pairs("t1")[:5])
print("T2 targets:", L.LOW)

**Takeaway:** T1 is 6,300 generations per model on the full validation set. That's too many for every experiment. While tuning we'll use a **subset** (e.g. 10 questions × ~12 pairs) and only run the full set for final numbers.

## Step 7 · How "language compliance" (LFS) is measured

**LFS = Language Forcing Success**: the fraction of answers that a fastText language-ID model labels as the target language, **with ≥ 0.9 confidence**. For zh/ja/ko there's a fallback that just looks at the script (Han / Kana / Hangul).

Let's poke at it to see what counts as a pass and what doesn't. *(Loading the model downloads ~1 GB once.)*

In [ ]:
from evaluate import load_lid, predict_lang, harmonic
lid = load_lid()
mapping = L.flores_to_code()

tests = {
    "clean German": "Die Photosynthese ist der Prozess, bei dem Pflanzen Licht in chemische Energie umwandeln.",
    "German + English mix": "Die Photosynthese ist ein process where plants convert light into energy, sehr wichtig.",
    "Norwegian": "Fotosyntese er prosessen der planter omdanner lys til kjemisk energi.",
    "short Chinese": "光合作用很重要。",
    "Japanese": "光合成とは、植物が光を化学エネルギーに変えるプロセスです。",
    "gibberish": "asdf qwer zxcv uiop",
}
for name, text in tests.items():
    labels, probs = lid.predict(" ".join(text.split()), k=2)
    top = [(l.replace("__label__", ""), round(float(p), 3)) for l, p in zip(labels, probs)]
    print(f"{name:22s} -> scored as '{predict_lang(lid, text, mapping)}'   raw top-2: {top}")

**What to notice:**
- Mixed-language text often falls below the 0.9 confidence bar and counts as a **fail** (`unk`). Half-steered answers get no partial credit.
- Close languages (Norwegian vs Danish/Swedish) can get confused, which is worth remembering for T2.
- The script fallback rescues short CJK text.

## Step 8 · The final score: LSS = harmonic mean of LFS and OR

- **LFS** (above): right language? In [0, 1].
- **OR = Output Relevance**: an LLM judge (Qwen3-8B) rates each answer 0/1/2 for "relevant, coherent, useful", normalized to [0, 1]. It is told to **ignore** the language.
- **LSS = 2·LFS·OR / (LFS + OR)**, the main leaderboard metric.

Harmonic mean punishes imbalance. Push steering so hard the model outputs German gibberish and LFS stays high, but OR crashes and so does LSS.

In [ ]:
cases = {"no steering (wrong language, good answer)": (0.0, 0.95),
         "over-steered (right language, gibberish)":  (1.0, 0.05),
         "balanced":                                  (0.8, 0.8),
         "lopsided":                                  (0.99, 0.5)}
for name, (lfs, orl) in cases.items():
    print(f"{name:45s} LFS={lfs:.2f} OR={orl:.2f} -> LSS={harmonic(lfs, orl):.3f}  (plain avg {(lfs+orl)/2:.3f})")

## Step 9 · Dry-run the official evaluator on fake outputs

Before we have a real system, let's create two fake "systems" in the **exact submission format** and score them with `evaluate.py --skip-judge` (LFS only; the judge needs a GPU):

1. **echo_source**: the "answer" is just the source-language question, i.e. what no steering looks like. We expect LFS ≈ 0.
2. **echo_target**: the "answer" is the *target-language version of the question*. Perfect language, but it isn't an answer, so LFS ≈ 1 and the judge would give a low OR.

This checks that the pipeline works on your machine and shows why LFS alone isn't enough.

In [ ]:
import subprocess
from clas.io_utils import write_json

demo_pairs = [("ar", "de"), ("en", "ja"), ("zh", "fr"), ("hi", "ru")]

for system in ["echo_source", "echo_target"]:
    out_dir = Path(f"generations/_sanity/{system}")
    for src, tgt in demo_pairs:
        outs = questions[src] if system == "echo_source" else questions[tgt]
        write_json(out_dir / f"{src}_to_{tgt}.json", {
            "source_language": src, "target_language": tgt,
            "model": "none", "method": system, "track": "t1",
            "results": [{"question_idx": i, "input": q, "output": o, "original_prompt": q}
                        for i, (q, o) in enumerate(zip(questions[src], outs))],
        })
    print(f"\n===== {system} =====")
    r = subprocess.run([sys.executable, "evaluate.py", "--gen-dir", str(out_dir), "--skip-judge"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("evaluate.py failed:\n", r.stderr[-2000:])
    else:
        print((out_dir / "scores.md").read_text())

**Reading the output:** OR and LSS show 0 because we skipped the judge. That's expected. Look at the **LFS** column: ≈0 for echo_source, ≈1 for echo_target.

## ✅ What you should understand now
1. **Input/output:** a question in language S goes in, an answer in language T comes out, with no instruction in the prompt.
2. **Data:** FLORES (parallel sentences) is for *finding* language directions. CLaS-Bench (70 parallel questions) is for *measuring*.
3. **Score:** LSS = harmonic mean of language-ID success and judge-rated relevance, so we need both.
4. **Format:** one `src_to_tgt.json` per pair, which our steering code must also produce.

**Next, notebook 02 (needs a GPU):** run the prompting baseline on a small subset to get a reference score, then extract our first language vectors from the model's hidden states.